# OCR Vintern-1B tren Kaggle - 2x GPU T4 (chay that, khong phai benchmark)

Notebook nay CHAY THAT toan bo keyframe qua OCR, chia 2 tien trinh song song tren 2 GPU T4.
Day ket qua len HuggingFace dinh ky de khong mat cong khi kernel bi loi.

**Truoc khi chay (Add Input):**
1. Dataset anh: `trnkhoa40phm/aic25-b1-keyframes-raw` (private) - mount tai
   `/kaggle/input/aic25-b1-keyframes-raw/L21/L21_V001/...`
2. Dataset chua `runtime.sqlite` (dataset rieng hoac gop chung voi anh - notebook tu do)

**Thu tu bat buoc (R1):** chay -> LUU (day HF + copy ket qua) -> KIEM (dem dong, assert).
Kernel Kaggle bi ERROR se xoa sach `/kaggle/working`, nen cell luu PHAI dung TRUOC cell kiem.

**Ngan sach thoi gian:** toi da 8h (tran phien nen Kaggle la 9h).

In [ ]:
# Cell 1 - Cai dat thu vien, ghim ban transformers trong chinh lenh cai (R6).
# Moc "<4.50" theo doc noi bo ve Vintern-1B/InternVL - CHO PHA 01 XAC NHAN LAI.
import subprocess
import sys

TRANSFORMERS_SPEC = "transformers>=4.37,<4.50"

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", TRANSFORMERS_SPEC],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError(f"Cai {TRANSFORMERS_SPEC} that bai - xem stderr o tren")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "einops", "timm", "sentencepiece", "huggingface_hub"],
    check=True,
)

import transformers
print(f"[CAI DAT] transformers.__version__ = {transformers.__version__}")
print(f"[CAI DAT] Moc spec da dung: {TRANSFORMERS_SPEC}")


In [ ]:
# Cell 2 - Lay HF_TOKEN tu Kaggle Secrets, KHONG hardcode.
# Thieu token van chay duoc, chi mat tinh nang day ket qua len HuggingFace.
import os

HF_TOKEN_OK = False
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    HF_TOKEN_OK = True
    print(f"[SECRETS] Da nhan HF_TOKEN tu Kaggle Secrets (do dai: {len(os.environ['HF_TOKEN'])} ky tu)")
except Exception as e:
    print(f"[SECRETS] [CANH BAO] Khong lay duoc HF_TOKEN: {e}")
    print("[SECRETS] [CANH BAO] Van chay OCR binh thuong, nhung se KHONG day duoc ket qua len HuggingFace.")
    print("[SECRETS] [CANH BAO] Vao Kaggle > Add-ons > Secrets, dat key 'HF_TOKEN' truoc khi chay lai neu can day.")


In [ ]:
# Cell 3 - Lay code moi nhat (R7): /kaggle/working duoc GIU giua cac lan chay,
# neu chi "if not exists() thi clone" se chay CODE CU. Xoa han roi clone lai.
import shutil
import subprocess
import sys as _sys

REPO_URL = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
REPO_BRANCH = "feat/ocr-kaggle-2gpu"
REPO_DIR = "/kaggle/working/repo"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_result = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR],
    capture_output=True, text=True,
)
if clone_result.returncode != 0:
    print(clone_result.stderr[-2000:])
    raise RuntimeError(f"Git clone that bai (nhanh {REPO_BRANCH}) - xem stderr o tren")

hash_result = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True,
)
commit_hash = hash_result.stdout.strip()
assert commit_hash, "Khong lay duoc commit hash - repo clone hong"
print(f"Commit: {commit_hash}")

# Xoa module cu (neu ton tai tu lan chay truoc trong cung kernel) de tranh dung code cache
for mod_name in list(_sys.modules.keys()):
    if "extract_ocr_vintern" in mod_name:
        del _sys.modules[mod_name]

SCRIPT_PATH = os.path.join(REPO_DIR, "system1", "research", "ocr_asr", "ocr", "extract_ocr_vintern.py")
assert os.path.exists(SCRIPT_PATH), f"Khong thay script tai {SCRIPT_PATH} - kiem tra nhanh {REPO_BRANCH}"
print(f"[CODE] Script chay: {SCRIPT_PATH}")


In [ ]:
# Cell 4 - Do /kaggle/input: tim thu muc anh (L21..L30) + runtime.sqlite, chep sqlite ra working.
# Bay da biet: sqlite ghi image_relpath = "keyframes/L21/L21_V001/xxx.jpg" nhung dataset
# THUC TE khong co thu muc bao "keyframes/" - anh nam thang <mount>/L21/L21_V001/xxx.jpg.
# Do ca 2 kieu roi kiem bang 1 file that truoc khi chot (giong kaggle_ocr_bench.ipynb cell 6).
import glob
import sqlite3

print(f"[DO] Cac thu muc trong /kaggle/input: {os.listdir('/kaggle/input')}")

sqlite_path_goc = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".sqlite"):
            sqlite_path_goc = os.path.join(root, f)
            break
    if sqlite_path_goc:
        break

if not sqlite_path_goc:
    raise FileNotFoundError(
        "Khong tim thay file .sqlite nao trong /kaggle/input.\n"
        "HUONG DAN: vao 'Add Input' > tim dataset chua runtime.sqlite (rieng hoac gop chung voi "
        "dataset anh trnkhoa40phm/aic25-b1-keyframes-raw) > Add > chay lai notebook."
    )
print(f"[DO] Tim thay sqlite goc: {sqlite_path_goc}")

DB_WORK_DIR = "/kaggle/working/db"
os.makedirs(DB_WORK_DIR, exist_ok=True)
DB_IN_PATH = os.path.join(DB_WORK_DIR, "runtime.sqlite")
shutil.copy(sqlite_path_goc, DB_IN_PATH)
print(f"[DO] Da chep sqlite ra ban doc: {DB_IN_PATH}")

# Doc 1 dong mau de do thu goc anh
conn = sqlite3.connect(DB_IN_PATH)
conn.row_factory = sqlite3.Row
mau = conn.execute("SELECT image_relpath FROM keyframes LIMIT 1").fetchone()
conn.close()
assert mau is not None, "Bang keyframes rong - kiem tra lai runtime.sqlite"
relpath_mau = mau["image_relpath"]
print(f"[DO] Relpath mau trong sqlite: {relpath_mau}")

def _tim_goc_anh(relpath_mau):
    """Thu ca 2 kieu duong dan (co/khong tien to 'keyframes/'), kiem bang 1 file that."""
    ten_khong_tien_to = relpath_mau
    if ten_khong_tien_to.startswith("keyframes/"):
        ten_khong_tien_to = ten_khong_tien_to[len("keyframes/"):]

    ung_vien_goc = []
    for root, dirs, _files in os.walk("/kaggle/input"):
        if os.path.basename(root).upper().startswith("L2") or os.path.basename(root).upper().startswith("L3"):
            ung_vien_goc.append(os.path.dirname(root))
        for d in dirs:
            if d == "keyframes":
                ung_vien_goc.append(os.path.join(root, d))

    for goc in ung_vien_goc:
        # thu duong dan nguyen ban (co the co "keyframes/")
        if os.path.exists(os.path.join(goc, relpath_mau)):
            return goc, False
        # thu bo tien to "keyframes/"
        if os.path.exists(os.path.join(goc, ten_khong_tien_to)):
            return goc, True
    return None, None

data_root, bo_tien_to = _tim_goc_anh(relpath_mau)
assert data_root is not None, (
    f"Khong tim thay anh mau '{relpath_mau}' (ca 2 kieu duong dan) duoi bat ky thu muc nao trong "
    "/kaggle/input. Kiem tra lai dataset anh da Add Input chua."
)
print(f"[DO] Thu muc goc anh: {data_root}")
print(f"[DO] Can bo tien to 'keyframes/' khi ghep duong dan: {bo_tien_to}")

DATA_ROOT = data_root


In [ ]:
# Cell 5 - CHAY: Popen 2 tien trinh song song (gpu 0 / gpu 1), python -u (R5) de log khong rong.
# Vong theo doi doc stdout ca 2, moi ~30 phut day ocr_part*.sqlite len HF (LUU DAN - xem ly do o markdown).
# Boc toan bo trong try/except - loi cung phai nhay xuong cell luu, khong duoc nem ra ngoai.
import time
import threading
import queue

import torch

NUM_GPUS_KHADUNG = torch.cuda.device_count()
print(f"[GPU] So GPU nhin thay: {NUM_GPUS_KHADUNG}")
for _i in range(NUM_GPUS_KHADUNG):
    print(f"[GPU] {_i}: {torch.cuda.get_device_name(_i)}")

NUM_GPUS = 2 if NUM_GPUS_KHADUNG >= 2 else 1
if NUM_GPUS == 1:
    print("[GPU] [CANH BAO] Chi thay 1 GPU - chay 1 tien trinh --num-gpus 1 --gpu-id 0 (khong crash).")

BATCH_SIZE = 4
DB_OUT_DIR = "/kaggle/working/db"
os.makedirs(DB_OUT_DIR, exist_ok=True)
HF_REPO_ID = "1thesudden/AIC26_checkpoints"
HF_UPLOAD_INTERVAL_SEC = 30 * 60

db_out_paths = [os.path.join(DB_OUT_DIR, f"ocr_part{i}.sqlite") for i in range(NUM_GPUS)]

def _doc_log_lien_tuc(proc, gpu_id, out_q):
    for line in iter(proc.stdout.readline, ""):
        if line:
            out_q.put((gpu_id, line.rstrip()))
    proc.stdout.close()

def _day_len_hf(paths, hf_token_ok):
    """Day ban sao cac file sqlite hien co len HF. Boc try/except - loi mang khong duoc lam chet job."""
    if not hf_token_ok:
        print("[HF] Bo qua day len HF - khong co HF_TOKEN")
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=os.environ.get("HF_TOKEN"))
        for p in paths:
            if not os.path.exists(p):
                continue
            ban_sao = p + ".upload_tmp"
            shutil.copy(p, ban_sao)
            ten_file = os.path.basename(p)
            api.upload_file(
                path_or_fileobj=ban_sao,
                path_in_repo=f"ocr/{ten_file}",
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                token=os.environ.get("HF_TOKEN"),
            )
            os.remove(ban_sao)
            print(f"[HF] Da day {ten_file} len {HF_REPO_ID}/ocr/")
    except Exception as e:
        print(f"[HF] [CANH BAO] Day HF that bai (khong lam chet job): {e}")

log_queue = queue.Queue()
procs = []
threads = []
try:
    for gpu_id in range(NUM_GPUS):
        cmd = [
            sys.executable, "-u", SCRIPT_PATH,
            "--gpu-id", str(gpu_id),
            "--num-gpus", str(NUM_GPUS),
            "--batch-size", str(BATCH_SIZE),
            "--db-in", DB_IN_PATH,
            "--db-out", db_out_paths[gpu_id],
            "--data-root", DATA_ROOT,
        ]
        print(f"[CHAY] Lenh GPU {gpu_id}: {' '.join(cmd)}")
        p = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        procs.append(p)
        t = threading.Thread(target=_doc_log_lien_tuc, args=(p, gpu_id, log_queue), daemon=True)
        t.start()
        threads.append(t)

    thoi_gian_bat_dau = time.time()
    lan_day_cuoi = time.time()
    while any(p.poll() is None for p in procs):
        try:
            gpu_id, dong = log_queue.get(timeout=5)
            print(f"[GPU{gpu_id}] {dong}")
        except queue.Empty:
            pass

        if time.time() - lan_day_cuoi >= HF_UPLOAD_INTERVAL_SEC:
            print(f"[HF] Da qua {HF_UPLOAD_INTERVAL_SEC // 60} phut - day dinh ky...")
            _day_len_hf(db_out_paths, HF_TOKEN_OK)
            lan_day_cuoi = time.time()

    # In not phan log con lai trong hang doi
    while not log_queue.empty():
        gpu_id, dong = log_queue.get()
        print(f"[GPU{gpu_id}] {dong}")

    ma_thoat = [p.returncode for p in procs]
    print(f"[CHAY] Ca {len(procs)} tien trinh da xong. Ma thoat: {ma_thoat}")
    for gpu_id, code in enumerate(ma_thoat):
        if code != 0:
            print(f"[CHAY] [CANH BAO] Tien trinh GPU {gpu_id} thoat voi ma loi {code} - phan da xong van duoc luu")

except Exception as e:
    print(f"[CHAY] [LOI] Ngoai le trong vong chay: {e}")
    print("[CHAY] Van tiep tuc xuong cell LUU de khong mat phan da lam.")
finally:
    thoi_gian_chay_giay = time.time() - thoi_gian_bat_dau if "thoi_gian_bat_dau" in dir() else 0.0
    print(f"[CHAY] Tong thoi gian chay: {thoi_gian_chay_giay/60:.1f} phut")


In [ ]:
# Cell 6 - LUU (R1: PHAI dung TRUOC cell KIEM). Day len HF lan cuoi + copy ket qua vao /kaggle/working.
print("[LUU] Day len HF lan cuoi...")
_day_len_hf(db_out_paths, HF_TOKEN_OK)

KET_QUA_DIR = "/kaggle/working/ket_qua"
os.makedirs(KET_QUA_DIR, exist_ok=True)
duong_dan_ket_qua = []
for p in db_out_paths:
    if os.path.exists(p):
        dich = os.path.join(KET_QUA_DIR, os.path.basename(p))
        shutil.copy(p, dich)
        duong_dan_ket_qua.append(dich)
        print(f"[LUU] Da copy {p} -> {dich}")
    else:
        print(f"[LUU] [CANH BAO] Khong thay file {p} de copy")

print(f"[LUU] Hoan tat. {len(duong_dan_ket_qua)}/{len(db_out_paths)} file da luu vao {KET_QUA_DIR}")


In [ ]:
# Cell 7 - KIEM (R1: dung SAU cell LUU). Dem dong tung part, tong, thoi gian, toc do, so loi.
print(f"Commit: {commit_hash}")
print(f"[KIEM] So GPU da dung: {NUM_GPUS}")

tong_dong = 0
bang_ket_qua = []
for gpu_id, p in enumerate(db_out_paths):
    if not os.path.exists(p):
        bang_ket_qua.append((gpu_id, 0, 0, "KHONG TON TAI"))
        continue
    conn = sqlite3.connect(p)
    try:
        so_dong_texts = conn.execute("SELECT COUNT(*) FROM ocr_texts").fetchone()[0]
        so_dong_fts = conn.execute("SELECT COUNT(*) FROM ocr_fts").fetchone()[0]
        trang_thai = "OK" if so_dong_texts == so_dong_fts else "LECH ocr_texts != ocr_fts"
    except Exception as e:
        so_dong_texts, so_dong_fts, trang_thai = 0, 0, f"LOI DOC: {e}"
    finally:
        conn.close()
    tong_dong += so_dong_texts
    bang_ket_qua.append((gpu_id, so_dong_texts, so_dong_fts, trang_thai))

print("\n[KIEM] Bang tong ket:")
print(f"{'gpu_id':>6} | {'ocr_texts':>10} | {'ocr_fts':>8} | trang_thai")
for gpu_id, n_texts, n_fts, trang_thai in bang_ket_qua:
    print(f"{gpu_id:>6} | {n_texts:>10} | {n_fts:>8} | {trang_thai}")

toc_do = tong_dong / thoi_gian_chay_giay if thoi_gian_chay_giay > 0 else 0.0
print(f"\n[KIEM] Tong so dong (ca 2 part): {tong_dong}")
print(f"[KIEM] Tong thoi gian chay: {thoi_gian_chay_giay/60:.1f} phut")
print(f"[KIEM] Toc do trung binh: {toc_do:.3f} anh/giay")
print(f"[KIEM] Ket qua da luu tai: {KET_QUA_DIR}")
print(f"[KIEM] Da day len HF: {HF_TOKEN_OK}")

assert tong_dong > 0, "Khong co dong nao duoc ghi - xem log GPU o Cell 5 de tim loi"
print("\n[KIEM] PASS")
